# 01 — Análise Exploratória (EDA)

**Dataset:** 2015 Flight Delays and Cancellations (U.S. DOT, ~5.8M voos).

**Objetivo deste notebook**:
1. Carregar e inspecionar o dataset com dtypes otimizados.
2. Estatísticas descritivas das variáveis numéricas e categóricas.
3. Análise de valores ausentes — decisões de tratamento.
4. Visualizações que respondem às perguntas-guia do desafio.
5. Conclusões que orientam a modelagem (notebooks 02-04).

**Perguntas que vamos responder**:
- Qual a distribuição do atraso? Qual % de voos atrasa ≥ 15 min?
- Quais aeroportos / companhias / dias / horários são mais críticos?
- Há padrão sazonal claro?
- Quais variáveis prometem como preditores?

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import config
from src.data_loader import describe_dataset, load_airlines, load_airports, load_flights, memory_usage_mb
from src.preprocessing import missing_report
from src import visualization as viz

sns.set_theme(style="whitegrid", palette="viridis")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("Configurações:")
print(f"  DATA_DIR     = {config.DATA_DIR}")
print(f"  SAMPLE_SIZE  = {config.SAMPLE_SIZE} (0 = dataset completo)")
print(f"  RANDOM_SEED  = {config.RANDOM_SEED}")

## 1. Carregamento dos dados

Usamos `load_flights` com dtypes explícitos — reduz uso de memória de ~3 GB para ~700 MB.

In [ ]:
flights = load_flights(sample_size=config.SAMPLE_SIZE)
airlines = load_airlines()
airports = load_airports()

print(f"flights:  {flights.shape[0]:>10,} linhas × {flights.shape[1]} colunas | {memory_usage_mb(flights):.1f} MB")
print(f"airlines: {airlines.shape}")
print(f"airports: {airports.shape}")

In [ ]:
flights.head()

In [ ]:
flights.describe(include="number").T.round(2)

**Observações da estatística descritiva**:
- `ARRIVAL_DELAY` tem média positiva mas mediana ~ –5 min → maioria dos voos chega antes ou no horário; outliers à direita puxam a média.
- `DEPARTURE_DELAY` e `ARRIVAL_DELAY` têm cauda longa (max > 1.000 min ≈ 17h).
- Colunas `*_DELAY` granulares (AIRLINE_DELAY, WEATHER_DELAY, etc.) têm muitos NaN porque só são preenchidas quando o voo atrasa ≥ 15 min.

## 2. Valores ausentes

In [ ]:
missing = missing_report(flights)
missing.head(15)

**Plano de tratamento** (implementado em `src.preprocessing.clean_pipeline`):

| Coluna(s) | Tratamento | Justificativa |
| :--- | :--- | :--- |
| `CANCELLATION_REASON` | descartar | só existe para voos cancelados; tarefa de cancelamento é fora do escopo |
| `*_DELAY` granulares | `fillna(0)` | NaN ≡ "sem contribuição" (voo não atrasou ≥ 15 min) |
| `TAIL_NUMBER` (~0.25%) | preencher `"UNKNOWN"` | manter linha; impacto pequeno |
| `ARRIVAL_DELAY`, `DEPARTURE_TIME` (~1.5%) | dropar | correspondem a voos cancelados/desviados; tarefa deles é outra |

In [ ]:
from src.preprocessing import clean_pipeline
from src.features import feature_pipeline

clean = clean_pipeline(flights)
clean = feature_pipeline(clean)
print(f"Após limpeza: {clean.shape[0]:>10,} linhas ({len(clean) / len(flights):.1%} do total)")

## 3. Variável alvo — `IS_DELAYED` (atraso ≥ 15 min na chegada)

In [ ]:
rate = clean["IS_DELAYED"].mean()
print(f"Taxa global de atraso (≥15 min): {rate:.2%}")
print(f"Atraso médio na chegada: {clean['ARRIVAL_DELAY'].mean():.1f} min")
print(f"Atraso mediano na chegada: {clean['ARRIVAL_DELAY'].median():.1f} min")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
viz.plot_delay_distribution(clean, "ARRIVAL_DELAY", clip=(-60, 180), ax=axes[0])
viz.plot_delay_distribution(clean[clean["ARRIVAL_DELAY"] > 0], "ARRIVAL_DELAY", clip=(0, 240), ax=axes[1])
axes[1].set_title("Distribuição (somente atrasos > 0)")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "01_arrival_delay_distribution.png", dpi=120)
plt.show()

## 4. Atraso por dimensões

### 4.1 Companhia aérea

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
viz.plot_delay_rate_by(clean, "AIRLINE", ax=ax)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "02_delay_by_airline.png", dpi=120)
plt.show()

### 4.2 Aeroporto de origem — top 25 piores

In [ ]:
airport_stats = (
    clean.groupby("ORIGIN_AIRPORT", observed=True)
    .agg(n_flights=("IS_DELAYED", "size"), delay_rate=("IS_DELAYED", "mean"))
    .query("n_flights >= 10000")
    .sort_values("delay_rate", ascending=False)
)
print("Top 10 aeroportos por taxa de atraso (≥10k voos no ano):")
airport_stats.head(10).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
top25 = airport_stats.head(25)
sns.barplot(x=top25.index, y=top25["delay_rate"] * 100, ax=ax, palette="rocket_r")
ax.set_title("Top 25 aeroportos por taxa de atraso (≥10k voos)")
ax.set_ylabel("% atrasos (≥15 min)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "03_delay_by_airport.png", dpi=120)
plt.show()

### 4.3 Heatmap dia da semana × hora do dia

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
viz.plot_delay_heatmap(clean, row="DAY_OF_WEEK", col="DEP_HOUR", ax=ax)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "04_heatmap_day_hour.png", dpi=120)
plt.show()

### 4.4 Sazonalidade — mês e estação do ano

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
viz.plot_delay_rate_by(clean, "MONTH", ax=axes[0])
axes[0].set_title("Taxa de atraso por mês")
viz.plot_delay_rate_by(clean, "SEASON", ax=axes[1])
axes[1].set_title("Taxa de atraso por estação")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "05_seasonality.png", dpi=120)
plt.show()

### 4.5 Feriados e fins de semana

In [ ]:
comp = pd.DataFrame({
    "Fim de semana": clean.groupby("IS_WEEKEND")["IS_DELAYED"].mean(),
    "Feriado federal": clean.groupby("IS_HOLIDAY")["IS_DELAYED"].mean(),
}).round(4) * 100
comp.index = ["Não", "Sim"]
print("Taxa de atraso (%):")
comp

## 5. Causas de atraso

Para os voos que efetivamente atrasaram (≥15 min), qual o peso médio de cada causa?

In [ ]:
delayed = clean[clean["IS_DELAYED"] == 1]
reason_share = (
    delayed[config.DELAY_REASON_COLS]
    .sum()
    .pipe(lambda s: s / s.sum() * 100)
    .round(2)
    .sort_values(ascending=False)
)
print("Contribuição percentual ao total de minutos atrasados:")
reason_share

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=reason_share.values, y=reason_share.index, palette="viridis", ax=ax)
ax.set_title("% do total de minutos atrasados, por causa")
ax.set_xlabel("% do total atrasado")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "06_delay_causes.png", dpi=120)
plt.show()

## 6. Correlação entre numéricas selecionadas

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
viz.plot_correlation_matrix(
    clean,
    cols=["ARRIVAL_DELAY", "DEPARTURE_DELAY", "DEP_HOUR", "DISTANCE", "SCHEDULED_TIME", "IS_WEEKEND", "IS_HOLIDAY"],
    ax=ax,
)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "07_correlation.png", dpi=120)
plt.show()

## 7. Conclusões da EDA

1. **Desbalanceamento moderado**: ~18-20% dos voos atrasam ≥ 15 min. Suficiente para predição, mas exige métricas além de accuracy (precision/recall, ROC-AUC, PR-AUC).
2. **Dimensões mais discriminativas** observadas: hora do dia (claro crescimento ao longo do dia), companhia aérea, aeroporto de origem, mês (jun/jul/dez piores) e feriados.
3. **Causa dominante**: atraso por *aircraft tardio* (efeito cascata) + companhia + NAS — o tempo agudo aparece concentrado, mas o efeito "propagação" domina.
4. **Vazamento a evitar**: `DEPARTURE_DELAY`, `TAXI_OUT`, `WHEELS_*`, `AIR_TIME` e os `*_DELAY` granulares só são conhecidos *após* o voo. Já listados em `config.LEAKY_COLS` e *não* serão usados como preditores nos notebooks 02/03.
5. **Features fortes para modelagem**: AIRLINE, ORIGIN_AIRPORT, DESTINATION_AIRPORT, DEP_HOUR, MONTH, DAY_OF_WEEK, IS_HOLIDAY, DISTANCE, SCHEDULED_TIME.

Próximos notebooks:
- **02_classification.ipynb** — predição binária `IS_DELAYED`.
- **03_regression.ipynb** — predição da duração do atraso.
- **04_unsupervised.ipynb** — clusterização de aeroportos + PCA.